In [1]:
!python -m pip install --no-index --no-deps /kaggle/input/datasets/daidysh643/gemma2-rm-finetune/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl

Processing /kaggle/input/datasets/daidysh643/gemma2-rm-finetune/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/gemma2_qlora_valid_predictions.csv
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter/adapter_model.safetensors
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter/training_args.bin
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter/adapter_config.json
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter/README.md
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter/tokenizer.json
/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapt

In [3]:
# ===== Path placeholders to update before submitting =====
COMPETITION_DATA_DIR = "/kaggle/input/competitions/llm-classification-finetuning"

# Upload the base model as a Kaggle Dataset, then replace this path.
BASE_MODEL_PATH = "/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/sfairXC__FsfairX-Gemma2-RM-v0.1"

# Upload output/gemma2_qlora_rm/adapter as a Kaggle Dataset, then replace this path.
ADAPTER_PATH = "/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter"

TEST_CSV = f"{COMPETITION_DATA_DIR}/test.csv"
SUBMISSION_CSV = "/kaggle/working/submission.csv"

# ===== Kaggle dual-T4 inference settings =====
MAX_LENGTH = 1800
BATCH_SIZE = 16  # per-GPU batch size when PARALLEL_INFERENCE=True
DTYPE = "float16"
LOAD_IN_4BIT = True
DEVICE_MAP = "auto"
TTA = False
DISABLE_SOFTCAPPING = True

# Data parallel inference: one process per GPU, automatic test split and merge.
PARALLEL_INFERENCE = True
NUM_INFERENCE_GPUS = None  # None uses all visible GPUs; set 2 for dual T4.
SHARD_DIR = "/kaggle/working/test_shards"
WORKER_SCRIPT_PATH = "/kaggle/working/parallel_infer_worker.py"

# Keep subprocess output out of the notebook cell to avoid interleaved progress bars.
SHOW_WORKER_OUTPUT = False
SHOW_TOTAL_PROGRESS = True
WORKER_LOG_DIR = f"{SHARD_DIR}/logs"

# If ADAPTER_PATH contains gemma2_qlora_config.json, these are read from it.
CLASSIFIER_HEAD = None  # "mlp" or "linear"
HEAD_DROPOUT = None
HEAD_HIDDEN_RATIO = None

# Optional quick smoke test. Set to None for the real submission.
LIMIT = None

In [4]:
import inspect
import json
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

torch 2.10.0+cu128
cuda available True
gpu Tesla T4


In [5]:
def parse_json_list(value):
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError("Expected a JSON list.")
    return ["" if item is None else str(item) for item in parsed]


def build_compact_pair_text(prompt_value, response_a_value, response_b_value):
    prompts = parse_json_list(prompt_value)
    responses_a = parse_json_list(response_a_value)
    responses_b = parse_json_list(response_b_value)

    turns = []
    for index, prompt in enumerate(prompts):
        response_a = responses_a[index] if index < len(responses_a) else ""
        response_b = responses_b[index] if index < len(responses_b) else ""
        turns.append(
            "<PROMPT>"
            + prompt.strip()
            + "</PROMPT><RESPONSE A>"
            + response_a.strip()
            + "</RESPONSE A><RESPONSE B>"
            + response_b.strip()
            + "</RESPONSE B>"
        )
    return "".join(turns)


def swap_dataframe(df):
    swapped = df.copy()
    swapped["response_a"], swapped["response_b"] = df["response_b"], df["response_a"]
    return swapped


class PreferenceDataset:
    def __init__(self, csv_path, tokenizer, max_length, limit=None, swap_inputs=False):
        df = pd.read_csv(csv_path)
        if limit is not None:
            df = df.head(limit).copy()
        if swap_inputs:
            df = swap_dataframe(df)

        required = ["id", "prompt", "response_a", "response_b"]
        missing = [column for column in required if column not in df.columns]
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {missing}")

        self.examples = [
            {
                "id": str(row["id"]),
                "text": build_compact_pair_text(row["prompt"], row["response_a"], row["response_b"]),
            }
            for _, row in df.iterrows()
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        encoded = self.tokenizer(
            example["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        encoded["id"] = example["id"]
        return encoded


class DataCollatorForPreference:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        ids = [feature.pop("id") for feature in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["id"] = ids
        return batch

In [6]:
def torch_dtype(dtype):
    if dtype == "float16":
        return torch.float16
    if dtype == "bfloat16":
        return torch.bfloat16
    if dtype == "float32":
        return torch.float32
    if dtype == "auto":
        return "auto"
    raise ValueError(f"Unsupported dtype: {dtype}")


def maybe_disable_softcapping(config, disable_softcapping):
    if disable_softcapping:
        if hasattr(config, "attn_logit_softcapping"):
            config.attn_logit_softcapping = None
        if hasattr(config, "final_logit_softcapping"):
            config.final_logit_softcapping = None
    return config


class MLPClassificationHead(nn.Module):
    def __init__(self, hidden_size, num_labels, dropout, hidden_ratio):
        super().__init__()
        head_hidden_size = max(num_labels, int(hidden_size * hidden_ratio))
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, head_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_size, num_labels),
        )

    def forward(self, hidden_states):
        return self.net(hidden_states)


def replace_classification_head(model, head_type, dropout, hidden_ratio):
    if head_type == "linear":
        return model
    if head_type != "mlp":
        raise ValueError(f"Unsupported classifier head: {head_type}")

    old_score_parameter = next(model.score.parameters(), None)
    device = old_score_parameter.device if old_score_parameter is not None else None
    dtype = old_score_parameter.dtype if old_score_parameter is not None else None
    new_score = MLPClassificationHead(model.config.hidden_size, model.config.num_labels, dropout, hidden_ratio)
    if device is not None and dtype is not None:
        new_score = new_score.to(device=device, dtype=dtype)
    model.score = new_score
    return model


@contextmanager
def peft_adapter_with_supported_config(adapter_path, config_cls):
    adapter_dir = Path(adapter_path)
    config_path = adapter_dir / "adapter_config.json"
    if not config_path.exists():
        yield adapter_path
        return

    adapter_config = json.loads(config_path.read_text(encoding="utf-8"))
    supported_keys = set(inspect.signature(config_cls.__init__).parameters)
    supported_keys.discard("self")
    filtered_config = {
        key: value
        for key, value in adapter_config.items()
        if key in supported_keys or key == "peft_type"
    }
    removed_keys = sorted(set(adapter_config) - set(filtered_config))
    if not removed_keys:
        yield adapter_path
        return

    with tempfile.TemporaryDirectory(prefix="peft_adapter_") as tmp:
        tmp_dir = Path(tmp)
        for item in adapter_dir.iterdir():
            target = tmp_dir / item.name
            if item.name == "adapter_config.json":
                continue
            os.symlink(item.resolve(), target, target_is_directory=item.is_dir())
        (tmp_dir / "adapter_config.json").write_text(
            json.dumps(filtered_config, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print("ignored unsupported PEFT keys:", ", ".join(removed_keys))
        yield str(tmp_dir)

In [7]:
def load_adapter_config_defaults():
    config_path = Path(ADAPTER_PATH) / "gemma2_qlora_config.json"
    saved_config = {}
    if config_path.exists():
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))

    classifier_head = CLASSIFIER_HEAD or saved_config.get("classifier_head", "mlp")
    head_dropout = HEAD_DROPOUT if HEAD_DROPOUT is not None else float(saved_config.get("head_dropout", 0.1))
    head_hidden_ratio = HEAD_HIDDEN_RATIO if HEAD_HIDDEN_RATIO is not None else float(saved_config.get("head_hidden_ratio", 0.5))
    return classifier_head, head_dropout, head_hidden_ratio


def load_tokenizer(model_path):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def load_model():
    from peft import LoraConfig, PeftModel
    from transformers import AutoConfig, AutoModelForSequenceClassification, BitsAndBytesConfig

    classifier_head, head_dropout, head_hidden_ratio = load_adapter_config_defaults()
    print("classifier_head", classifier_head)
    print("head_dropout", head_dropout)
    print("head_hidden_ratio", head_hidden_ratio)

    config = AutoConfig.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
    config.num_labels = len(LABEL_COLUMNS)
    config = maybe_disable_softcapping(config, DISABLE_SOFTCAPPING)

    quantization_config = None
    device_map = DEVICE_MAP if LOAD_IN_4BIT else None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype(DTYPE),
            bnb_4bit_use_double_quant=True,
        )
        print("device_map", device_map)

    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_PATH,
        config=config,
        trust_remote_code=True,
        torch_dtype=torch_dtype(DTYPE),
        quantization_config=quantization_config,
        device_map=device_map,
        ignore_mismatched_sizes=True,
    )
    if base.config.pad_token_id is None:
        base.config.pad_token_id = base.config.eos_token_id
    base.config = maybe_disable_softcapping(base.config, DISABLE_SOFTCAPPING)
    base = replace_classification_head(base, classifier_head, head_dropout, head_hidden_ratio)

    with peft_adapter_with_supported_config(ADAPTER_PATH, LoraConfig) as adapter_path:
        model = PeftModel.from_pretrained(base, adapter_path)
    if not LOAD_IN_4BIT and torch.cuda.is_available():
        model = model.to("cuda")
    model.eval()
    return model

In [8]:
def model_input_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def softmax(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def predict_dataset(model, dataset, tokenizer, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=DataCollatorForPreference(tokenizer),
    )
    ids = []
    probabilities = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="predict"):
            row_ids = batch.pop("id")
            device = model_input_device(model)
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch).logits.detach().float().cpu().numpy()
            ids.extend(row_ids)
            probabilities.extend(softmax(logits))
    return ids, probabilities

In [9]:
import math
import subprocess
import sys
import time
from pathlib import Path

WORKER_SCRIPT = r'''
import argparse
import inspect
import json
import math
import os
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    from transformers.utils import logging as transformers_logging

    transformers_logging.set_verbosity_error()
    transformers_logging.disable_progress_bar()
except Exception:
    pass

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]


def parse_json_list(value):
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError("Expected a JSON list.")
    return ["" if item is None else str(item) for item in parsed]


def build_compact_pair_text(prompt_value, response_a_value, response_b_value):
    prompts = parse_json_list(prompt_value)
    responses_a = parse_json_list(response_a_value)
    responses_b = parse_json_list(response_b_value)

    turns = []
    for index, prompt in enumerate(prompts):
        response_a = responses_a[index] if index < len(responses_a) else ""
        response_b = responses_b[index] if index < len(responses_b) else ""
        turns.append(
            "<PROMPT>"
            + prompt.strip()
            + "</PROMPT><RESPONSE A>"
            + response_a.strip()
            + "</RESPONSE A><RESPONSE B>"
            + response_b.strip()
            + "</RESPONSE B>"
        )
    return "".join(turns)


def swap_dataframe(df):
    swapped = df.copy()
    swapped["response_a"], swapped["response_b"] = df["response_b"], df["response_a"]
    return swapped


class PreferenceDataset:
    def __init__(self, csv_path, tokenizer, max_length, swap_inputs=False):
        df = pd.read_csv(csv_path)
        if swap_inputs:
            df = swap_dataframe(df)

        required = ["id", "prompt", "response_a", "response_b", "__row_order"]
        missing = [column for column in required if column not in df.columns]
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {missing}")

        self.examples = [
            {
                "id": str(row["id"]),
                "row_order": int(row["__row_order"]),
                "text": build_compact_pair_text(row["prompt"], row["response_a"], row["response_b"]),
            }
            for _, row in df.iterrows()
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        encoded = self.tokenizer(
            example["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        encoded["id"] = example["id"]
        encoded["row_order"] = example["row_order"]
        return encoded


class DataCollatorForPreference:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        ids = [feature.pop("id") for feature in features]
        row_orders = [feature.pop("row_order") for feature in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["id"] = ids
        batch["row_order"] = row_orders
        return batch


def torch_dtype(dtype):
    if dtype == "float16":
        return torch.float16
    if dtype == "bfloat16":
        return torch.bfloat16
    if dtype == "float32":
        return torch.float32
    if dtype == "auto":
        return "auto"
    raise ValueError(f"Unsupported dtype: {dtype}")


def maybe_disable_softcapping(config, disable_softcapping):
    if disable_softcapping:
        if hasattr(config, "attn_logit_softcapping"):
            config.attn_logit_softcapping = None
        if hasattr(config, "final_logit_softcapping"):
            config.final_logit_softcapping = None
    return config


class MLPClassificationHead(nn.Module):
    def __init__(self, hidden_size, num_labels, dropout, hidden_ratio):
        super().__init__()
        head_hidden_size = max(num_labels, int(hidden_size * hidden_ratio))
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, head_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_size, num_labels),
        )

    def forward(self, hidden_states):
        return self.net(hidden_states)


def replace_classification_head(model, head_type, dropout, hidden_ratio):
    if head_type == "linear":
        return model
    if head_type != "mlp":
        raise ValueError(f"Unsupported classifier head: {head_type}")

    old_score_parameter = next(model.score.parameters(), None)
    device = old_score_parameter.device if old_score_parameter is not None else None
    dtype = old_score_parameter.dtype if old_score_parameter is not None else None
    new_score = MLPClassificationHead(model.config.hidden_size, model.config.num_labels, dropout, hidden_ratio)
    if device is not None and dtype is not None:
        new_score = new_score.to(device=device, dtype=dtype)
    model.score = new_score
    return model


@contextmanager
def peft_adapter_with_supported_config(adapter_path, config_cls):
    adapter_dir = Path(adapter_path)
    config_path = adapter_dir / "adapter_config.json"
    if not config_path.exists():
        yield adapter_path
        return

    adapter_config = json.loads(config_path.read_text(encoding="utf-8"))
    supported_keys = set(inspect.signature(config_cls.__init__).parameters)
    supported_keys.discard("self")
    filtered_config = {
        key: value
        for key, value in adapter_config.items()
        if key in supported_keys or key == "peft_type"
    }
    removed_keys = sorted(set(adapter_config) - set(filtered_config))
    if not removed_keys:
        yield adapter_path
        return

    with tempfile.TemporaryDirectory(prefix="peft_adapter_") as tmp:
        tmp_dir = Path(tmp)
        for item in adapter_dir.iterdir():
            target = tmp_dir / item.name
            if item.name == "adapter_config.json":
                continue
            os.symlink(item.resolve(), target, target_is_directory=item.is_dir())
        (tmp_dir / "adapter_config.json").write_text(
            json.dumps(filtered_config, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print("ignored unsupported PEFT keys:", ", ".join(removed_keys))
        yield str(tmp_dir)


def load_adapter_config_defaults(cfg):
    config_path = Path(cfg["adapter_path"]) / "gemma2_qlora_config.json"
    saved_config = {}
    if config_path.exists():
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))

    classifier_head = cfg["classifier_head"] or saved_config.get("classifier_head", "mlp")
    head_dropout = cfg["head_dropout"] if cfg["head_dropout"] is not None else float(saved_config.get("head_dropout", 0.1))
    head_hidden_ratio = cfg["head_hidden_ratio"] if cfg["head_hidden_ratio"] is not None else float(saved_config.get("head_hidden_ratio", 0.5))
    return classifier_head, head_dropout, head_hidden_ratio


def load_tokenizer(model_path):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def load_model(cfg):
    from peft import LoraConfig, PeftModel
    from transformers import AutoConfig, AutoModelForSequenceClassification, BitsAndBytesConfig

    classifier_head, head_dropout, head_hidden_ratio = load_adapter_config_defaults(cfg)
    print("classifier_head", classifier_head)
    print("head_dropout", head_dropout)
    print("head_hidden_ratio", head_hidden_ratio)

    config = AutoConfig.from_pretrained(cfg["base_model_path"], trust_remote_code=True)
    config.num_labels = len(LABEL_COLUMNS)
    config = maybe_disable_softcapping(config, cfg["disable_softcapping"])

    quantization_config = None
    device_map = cfg["device_map"] if cfg["load_in_4bit"] else None
    if cfg["load_in_4bit"]:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype(cfg["dtype"]),
            bnb_4bit_use_double_quant=True,
        )
        print("device_map", device_map)

    base = AutoModelForSequenceClassification.from_pretrained(
        cfg["base_model_path"],
        config=config,
        trust_remote_code=True,
        torch_dtype=torch_dtype(cfg["dtype"]),
        quantization_config=quantization_config,
        device_map=device_map,
        ignore_mismatched_sizes=True,
    )
    if base.config.pad_token_id is None:
        base.config.pad_token_id = base.config.eos_token_id
    base.config = maybe_disable_softcapping(base.config, cfg["disable_softcapping"])
    base = replace_classification_head(base, classifier_head, head_dropout, head_hidden_ratio)

    with peft_adapter_with_supported_config(cfg["adapter_path"], LoraConfig) as adapter_path:
        model = PeftModel.from_pretrained(base, adapter_path)
    if not cfg["load_in_4bit"] and torch.cuda.is_available():
        model = model.to("cuda")
    model.eval()
    return model


def model_input_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def softmax(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def write_progress(progress_file, done_batches, total_batches):
    if progress_file is None:
        return
    Path(progress_file).write_text(
        json.dumps({"done": done_batches, "total": total_batches}),
        encoding="utf-8",
    )


def predict_dataset(
    model,
    dataset,
    tokenizer,
    batch_size,
    desc,
    disable_tqdm=False,
    progress_file=None,
    progress_start=0,
    progress_total=None,
):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=DataCollatorForPreference(tokenizer),
        pin_memory=torch.cuda.is_available(),
    )
    ids = []
    row_orders = []
    probabilities = []
    done_batches = progress_start
    total_batches = len(loader) if progress_total is None else progress_total
    write_progress(progress_file, done_batches, total_batches)
    with torch.inference_mode():
        for batch in tqdm(loader, desc=desc, disable=disable_tqdm):
            row_ids = batch.pop("id")
            batch_row_orders = batch.pop("row_order")
            device = model_input_device(model)
            batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
            logits = model(**batch).logits.detach().float().cpu().numpy()
            ids.extend(row_ids)
            row_orders.extend(batch_row_orders)
            probabilities.extend(softmax(logits))
            done_batches += 1
            write_progress(progress_file, done_batches, total_batches)
    return ids, row_orders, probabilities


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--shard-index", type=int, required=True)
    parser.add_argument("--ready-file", required=True)
    parser.add_argument("--start-file", required=True)
    parser.add_argument("--progress-file")
    args = parser.parse_args()

    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))
    print(f"worker shard={args.shard_index} visible_gpus={os.environ.get('CUDA_VISIBLE_DEVICES')}")

    tokenizer = load_tokenizer(cfg["adapter_path"])
    model = load_model(cfg)

    ready_file = Path(args.ready_file)
    start_file = Path(args.start_file)
    ready_file.parent.mkdir(parents=True, exist_ok=True)
    ready_file.write_text("ready\n", encoding="utf-8")
    print(f"worker shard={args.shard_index} model loaded; waiting for {start_file}")
    while not start_file.exists():
        time.sleep(1)
    print(f"worker shard={args.shard_index} starting inference")

    dataset = PreferenceDataset(args.input, tokenizer, max_length=cfg["max_length"])
    original_batches = math.ceil(len(dataset) / cfg["batch_size"])
    total_batches = original_batches * (2 if cfg["tta"] else 1)
    ids, row_orders, probabilities = predict_dataset(
        model,
        dataset,
        tokenizer,
        cfg["batch_size"],
        desc=f"predict shard {args.shard_index}",
        disable_tqdm=cfg["disable_worker_tqdm"],
        progress_file=args.progress_file,
        progress_start=0,
        progress_total=total_batches,
    )

    if cfg["tta"]:
        swapped_dataset = PreferenceDataset(
            args.input,
            tokenizer,
            max_length=cfg["max_length"],
            swap_inputs=True,
        )
        swapped_ids, swapped_row_orders, swapped_probabilities = predict_dataset(
            model,
            swapped_dataset,
            tokenizer,
            cfg["batch_size"],
            desc=f"tta shard {args.shard_index}",
            disable_tqdm=cfg["disable_worker_tqdm"],
            progress_file=args.progress_file,
            progress_start=original_batches,
            progress_total=total_batches,
        )
        if ids != swapped_ids or row_orders != swapped_row_orders:
            raise ValueError("Original and swapped prediction rows do not match.")
        probabilities = [
            (original + swapped[[1, 0, 2]]) / 2.0
            for original, swapped in zip(probabilities, swapped_probabilities)
        ]

    rows = []
    for row_id, row_order, probs in zip(ids, row_orders, probabilities):
        rows.append(
            {
                "__row_order": row_order,
                "id": row_id,
                LABEL_COLUMNS[0]: probs[0],
                LABEL_COLUMNS[1]: probs[1],
                LABEL_COLUMNS[2]: probs[2],
            }
        )
    output = Path(args.output)
    output.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(output, index=False)
    print(f"Wrote {output}")


if __name__ == "__main__":
    main()


'''

def write_worker_script():
    worker_path = Path(WORKER_SCRIPT_PATH)
    worker_path.parent.mkdir(parents=True, exist_ok=True)
    worker_path.write_text(WORKER_SCRIPT, encoding="utf-8")
    return worker_path


def build_parallel_config():
    return {
        "base_model_path": BASE_MODEL_PATH,
        "adapter_path": ADAPTER_PATH,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "dtype": DTYPE,
        "load_in_4bit": LOAD_IN_4BIT,
        "device_map": DEVICE_MAP,
        "tta": TTA,
        "disable_softcapping": DISABLE_SOFTCAPPING,
        "classifier_head": CLASSIFIER_HEAD,
        "head_dropout": HEAD_DROPOUT,
        "head_hidden_ratio": HEAD_HIDDEN_RATIO,
        "disable_worker_tqdm": not SHOW_WORKER_OUTPUT,
    }


def split_test_csv(num_shards):
    shard_dir = Path(SHARD_DIR)
    shard_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(TEST_CSV)
    if LIMIT is not None:
        df = df.head(LIMIT).copy()
    df = df.copy()
    df["__row_order"] = np.arange(len(df), dtype=np.int64)

    shard_paths = []
    shard_row_counts = []
    for shard_index in range(num_shards):
        shard = df.iloc[shard_index::num_shards].copy()
        shard_path = shard_dir / f"test_shard_{shard_index}.csv"
        shard.to_csv(shard_path, index=False)
        shard_paths.append(shard_path)
        shard_row_counts.append(len(shard))
        print(f"shard {shard_index}: {len(shard)} rows -> {shard_path}")
    return shard_paths, len(df), shard_row_counts


def run_parallel_inference():
    visible_gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    requested_gpu_count = NUM_INFERENCE_GPUS or visible_gpu_count
    num_shards = max(1, min(requested_gpu_count, visible_gpu_count))
    if num_shards < 2:
        print("Less than 2 visible GPUs; falling back to single-process inference.")
        return None

    worker_path = write_worker_script()
    shard_paths, total_rows, shard_row_counts = split_test_csv(num_shards)

    config_path = Path(SHARD_DIR) / "parallel_config.json"
    config_path.write_text(json.dumps(build_parallel_config(), indent=2), encoding="utf-8")

    start_file = Path(SHARD_DIR) / "start_inference.signal"
    if start_file.exists():
        start_file.unlink()

    processes = []
    output_paths = []
    progress_paths = []
    log_handles = []
    for shard_index, shard_path in enumerate(shard_paths):
        output_path = Path(SHARD_DIR) / f"pred_shard_{shard_index}.csv"
        ready_file = Path(SHARD_DIR) / f"model_ready_{shard_index}.signal"
        if ready_file.exists():
            ready_file.unlink()
        output_paths.append(output_path)
        progress_file = Path(SHARD_DIR) / f"progress_shard_{shard_index}.json"
        if progress_file.exists():
            progress_file.unlink()
        progress_paths.append(progress_file)
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(shard_index)
        command = [
            sys.executable,
            str(worker_path),
            "--config",
            str(config_path),
            "--input",
            str(shard_path),
            "--output",
            str(output_path),
            "--shard-index",
            str(shard_index),
            "--ready-file",
            str(ready_file),
            "--start-file",
            str(start_file),
            "--progress-file",
            str(progress_file),
        ]
        stdout = None
        stderr = None
        if not SHOW_WORKER_OUTPUT:
            log_dir = Path(WORKER_LOG_DIR)
            log_dir.mkdir(parents=True, exist_ok=True)
            log_handle = (log_dir / f"worker_{shard_index}.log").open("w", encoding="utf-8")
            stdout = log_handle
            stderr = subprocess.STDOUT
            log_handles.append(log_handle)
        print("loading model", shard_index, "on physical GPU", shard_index)
        process = subprocess.Popen(command, env=env, stdout=stdout, stderr=stderr)
        processes.append((shard_index, process))

    with tqdm(total=num_shards, desc="load models", disable=not SHOW_TOTAL_PROGRESS, unit="model") as load_bar:
        loaded = set()
        while len(loaded) < num_shards:
            for shard_index, process in processes:
                ready_file = Path(SHARD_DIR) / f"model_ready_{shard_index}.signal"
                if shard_index in loaded:
                    continue
                return_code = process.poll()
                if return_code is not None:
                    raise RuntimeError(f"Model loading failed on shard {shard_index} with code {return_code}")
                if ready_file.exists():
                    loaded.add(shard_index)
                    load_bar.update(1)
            if len(loaded) < num_shards:
                time.sleep(2)

    print("all models loaded; starting parallel inference")
    start_file.write_text("start\n", encoding="utf-8")

    total_batches = sum(math.ceil(count / BATCH_SIZE) for count in shard_row_counts) * (2 if TTA else 1)
    with tqdm(total=total_batches, desc="parallel inference", disable=not SHOW_TOTAL_PROGRESS, unit="batch") as infer_bar:
        while processes:
            done_batches = 0
            for progress_path in progress_paths:
                if progress_path.exists():
                    try:
                        done_batches += int(json.loads(progress_path.read_text(encoding="utf-8")).get("done", 0))
                    except json.JSONDecodeError:
                        pass
            infer_bar.update(max(0, done_batches - infer_bar.n))

            still_running = []
            for shard_index, process in processes:
                return_code = process.poll()
                if return_code is None:
                    still_running.append((shard_index, process))
                elif return_code != 0:
                    for _, running_process in still_running:
                        running_process.terminate()
                    raise RuntimeError(f"Parallel inference failed on shard {shard_index} with code {return_code}")
            if still_running:
                time.sleep(5)
            processes = still_running
        infer_bar.update(max(0, total_batches - infer_bar.n))

    for log_handle in log_handles:
        log_handle.close()

    parts = [pd.read_csv(path) for path in output_paths]
    submission = pd.concat(parts, ignore_index=True)
    submission = submission.sort_values("__row_order").drop(columns="__row_order").reset_index(drop=True)
    if len(submission) != total_rows:
        raise ValueError(f"Expected {total_rows} predictions, got {len(submission)}")
    submission.to_csv(SUBMISSION_CSV, index=False)
    print(f"Wrote {SUBMISSION_CSV}")
    return submission


if PARALLEL_INFERENCE:
    submission = run_parallel_inference()
else:
    submission = None

if submission is None:
    tokenizer = load_tokenizer(ADAPTER_PATH)
    model = load_model()

    dataset = PreferenceDataset(TEST_CSV, tokenizer, max_length=MAX_LENGTH, limit=LIMIT)
    ids, probabilities = predict_dataset(model, dataset, tokenizer, BATCH_SIZE)

    if TTA:
        swapped_dataset = PreferenceDataset(
            TEST_CSV,
            tokenizer,
            max_length=MAX_LENGTH,
            limit=LIMIT,
            swap_inputs=True,
        )
        swapped_ids, swapped_probabilities = predict_dataset(model, swapped_dataset, tokenizer, BATCH_SIZE)
        if ids != swapped_ids:
            raise ValueError("Original and swapped prediction ids do not match.")
        probabilities = [
            (original + swapped[[1, 0, 2]]) / 2.0
            for original, swapped in zip(probabilities, swapped_probabilities)
        ]

    submission = pd.DataFrame(
        {
            "id": ids,
            LABEL_COLUMNS[0]: [probs[0] for probs in probabilities],
            LABEL_COLUMNS[1]: [probs[1] for probs in probabilities],
            LABEL_COLUMNS[2]: [probs[2] for probs in probabilities],
        }
    )
    submission.to_csv(SUBMISSION_CSV, index=False)
    print(f"Wrote {SUBMISSION_CSV}")

submission.head()

shard 0: 2 rows -> /kaggle/working/test_shards/test_shard_0.csv
shard 1: 1 rows -> /kaggle/working/test_shards/test_shard_1.csv
loading model 0 on physical GPU 0
worker shard=0 visible_gpus=0
classifier_head mlp
head_dropout 0.1
head_hidden_ratio 0.5
device_map auto


Loading weights: 100%|██████████| 465/465 [00:29<00:00, 15.52it/s, Materializing param=score.weight]                                     


model 0 loaded and waiting
loading model 1 on physical GPU 1


Loading weights:   0%|          | 1/465 [00:00<00:00, 4128.25it/s, Materializing param=model.embed_tokens.weight] 

worker shard=1 visible_gpus=1
classifier_head mlp
head_dropout 0.1
head_hidden_ratio 0.5
device_map auto


...

predict shard 1: 100%|██████████| 1/1 [00:01<00:00,  1.68s/it]


Wrote /kaggle/working/test_shards/pred_shard_1.csv


predict shard 0: 100%|██████████| 1/1 [00:04<00:00,  4.30s/it]


Wrote /kaggle/working/test_shards/pred_shard_0.csv
Wrote /kaggle/working/submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.003733,0.974312,0.021955
1,211333,0.510251,0.190063,0.299687
2,1233961,0.170751,0.529041,0.300208


In [10]:
submission = pd.read_csv(SUBMISSION_CSV)
expected_columns = ["id", "winner_model_a", "winner_model_b", "winner_tie"]
assert list(submission.columns) == expected_columns, submission.columns.tolist()
assert submission[expected_columns[1:]].notna().all().all()
assert np.isfinite(submission[expected_columns[1:]].to_numpy()).all()
print(submission.shape)
print(submission[expected_columns[1:]].sum(axis=1).describe())
submission.head()

(3, 4)
count    3.000000e+00
mean     1.000000e+00
std      3.775398e-08
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.003733,0.974312,0.021955
1,211333,0.510251,0.190063,0.299687
2,1233961,0.170751,0.529041,0.300208
